# Taller 2 — DQN sobre Pong (Atari)

**Simulación y Aprendizaje por Refuerzo** — Maestría en IA, Universidad de La Sabana.  
Autor: Ivan Enrique Rangel Santos.

Notebook autónomo. Antes de ejecutar: **Runtime → Change runtime type → T4 GPU**.

Duración esperada: ~1.5–2 h de entrenamiento en T4 para 1.5 M pasos. Colab gratuito puede cortar la sesión — la celda de entrenamiento guarda checkpoints cada 100 k pasos para poder reanudar.

## 1. Instalación de dependencias

In [ ]:
!pip install -q "gymnasium[atari]==1.3.0" "ale-py>=0.11" "torch" "numpy<2" "matplotlib" "imageio[ffmpeg]"

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Wrappers de preprocesamiento (Atari)

El frame crudo de Atari es 210×160 RGB a 60 Hz. Meter eso a una red desperdicia cómputo y oculta la señal útil. Los wrappers convierten la observación en algo que una CNN puede aprender.

| Wrapper | Qué hace | Por qué |
|---|---|---|
| `noop_max=30` | Ejecuta 0–30 no-ops al reset | Aleatoriza el estado inicial; el agente no puede memorizar |
| `frame_skip=4` | El agente decide cada 4 frames; el env repite la acción | Estándar Atari; el juego es demasiado rápido para decidir cada frame |
| (max sobre 2 frames) | Toma el max pixel-wise de los últimos 2 frames del skip | Defeats sprite flicker (Atari muestra sprites en frames alternos) |
| `grayscale + 84×84` | RGB→grayscale, resize a 84×84 | 3× menos entrada, ninguna señal útil se pierde para Pong |
| `FrameStackObservation(4)` | Apila los últimos 4 frames procesados | Un frame dice DÓNDE está la bola, no HACIA DÓNDE va |

Salida final: `(4, 84, 84)` `uint8` — layout `(canales, alto, ancho)` que PyTorch espera.

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
import ale_py


def make_atari_env(env_id='ALE/Pong-v5', seed=None, render_mode=None):
    gym.register_envs(ale_py)
    env = gym.make(env_id, frameskip=1, render_mode=render_mode)
    env = AtariPreprocessing(
        env,
        noop_max=30,
        frame_skip=4,
        screen_size=84,
        terminal_on_life_loss=False,
        grayscale_obs=True,
        grayscale_newaxis=False,
        scale_obs=False,
    )
    env = FrameStackObservation(env, stack_size=4)
    if seed is not None:
        env.reset(seed=seed)
        env.action_space.seed(seed)
    return env


# Inspección rápida
env = make_atari_env(seed=0)
obs, _ = env.reset(seed=0)
import numpy as np
obs = np.asarray(obs)
print('obs shape:', obs.shape, 'dtype:', obs.dtype, 'range:', obs.min(), obs.max())
print('action space:', env.action_space, 'meanings:', env.unwrapped.get_action_meanings())
env.close()

## 3. Red neuronal — Nature CNN (Mnih et al., 2015)

```
input  (batch, 4, 84, 84) uint8      # 4 canales = frame stack, valores 0-255
  → normalizar /255 → float32 (dentro del forward, para ahorrar memoria)
  Conv2d(4→32,  kernel=8, stride=4)  ReLU   →  (batch, 32, 20, 20)
  Conv2d(32→64, kernel=4, stride=2)  ReLU   →  (batch, 64,  9,  9)
  Conv2d(64→64, kernel=3, stride=1)  ReLU   →  (batch, 64,  7,  7)
  Flatten                                   →  (batch, 3136)
  Linear(3136→512)                   ReLU   →  (batch, 512)
  Linear(512→6)                             →  (batch, 6)   # 1 Q por acción
```

**Por qué esto:**
- Kernel 8 + stride 4 al inicio: campo receptivo grande por pixel. La bola es pequeña pero su trayectoria abarca la pantalla.
- Stride decreciente (4 → 2 → 1): coarse-to-fine. Primero capturar movimiento grueso, después estructura local.
- Sin pooling: los strides ya subsamplean, y pooling tiraría precisión espacial que la paleta necesita.
- Sin BatchNorm: los targets de DQN son no-estacionarios, BatchNorm es inestable en ese régimen (resultado bien conocido).
- Cabeza lineal: un Q por cada una de las 6 acciones (NOOP, FIRE, RIGHT, LEFT, RIGHTFIRE, LEFTFIRE).

In [ ]:
import torch
import torch.nn as nn


class NatureCNN(nn.Module):
    def __init__(self, in_channels=4, n_actions=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 8, 4), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, 2),          nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, 1),          nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Linear(3136, 512), nn.ReLU(inplace=True),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        if x.dtype == torch.uint8:
            x = x.float() / 255.0
        return self.head(self.features(x))


# Sanity check de forma
m = NatureCNN()
print('parámetros:', sum(p.numel() for p in m.parameters()))
print('forward:', m(torch.zeros(2, 4, 84, 84, dtype=torch.uint8)).shape)

## 4. Replay buffer con almacenamiento `uint8`

Cada observación es `(4, 84, 84)` en `uint8` = 28 224 bytes ≈ 28 KB.  
Con `capacity=100_000` y guardando obs + next_obs por separado, el buffer ocupa ~5.6 GB de RAM. Cabe holgado en Colab T4 (12–15 GB de RAM del host).

In [ ]:
from collections import namedtuple

Transition = namedtuple('Transition', 'obs actions rewards next_obs dones')

class ReplayBuffer:
    def __init__(self, capacity, obs_shape, device='cpu'):
        self.capacity = int(capacity); self.device = device
        self.obs      = np.zeros((capacity, *obs_shape), dtype=np.uint8)
        self.next_obs = np.zeros((capacity, *obs_shape), dtype=np.uint8)
        self.actions  = np.zeros(capacity, dtype=np.int64)
        self.rewards  = np.zeros(capacity, dtype=np.float32)
        self.dones    = np.zeros(capacity, dtype=np.float32)
        self._idx = 0; self._size = 0

    def __len__(self): return self._size

    def push(self, obs, a, r, next_obs, done):
        i = self._idx
        self.obs[i] = obs; self.next_obs[i] = next_obs
        self.actions[i] = a; self.rewards[i] = r; self.dones[i] = float(done)
        self._idx = (i + 1) % self.capacity
        self._size = min(self._size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self._size, size=batch_size)
        d = self.device
        return Transition(
            torch.from_numpy(self.obs[idx]).to(d, non_blocking=True),
            torch.from_numpy(self.actions[idx]).to(d, non_blocking=True),
            torch.from_numpy(self.rewards[idx]).to(d, non_blocking=True),
            torch.from_numpy(self.next_obs[idx]).to(d, non_blocking=True),
            torch.from_numpy(self.dones[idx]).to(d, non_blocking=True),
        )

## 5. Agente DQN

Encapsula: red online + red target, ε-greedy con decay lineal, actualización de Bellman con MSE Huber (`smooth_l1`), clip de gradiente.

El **Huber loss** es lo estándar para DQN — es cuadrática cerca de 0 (como MSE) pero lineal fuera, así que un target ocasionalmente enorme no vuela el gradiente.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
import torch.nn.functional as F


@dataclass
class DQNConfig:
    env_id: str = 'ALE/Pong-v5'
    n_actions: int = 6
    frame_stack: int = 4
    lr: float = 2.5e-4
    gamma: float = 0.99
    batch_size: int = 32
    max_grad_norm: float = 10.0
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_decay_steps: int = 250_000
    buffer_capacity: int = 100_000
    replay_start_size: int = 10_000
    learn_every: int = 4
    target_update_every: int = 1_000
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed: int = 0


class DQNAgent:
    def __init__(self, cfg):
        self.cfg = cfg
        torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
        self.q_net      = NatureCNN(cfg.frame_stack, cfg.n_actions).to(cfg.device)
        self.target_net = deepcopy(self.q_net).to(cfg.device)
        for p in self.target_net.parameters(): p.requires_grad = False
        self.optimizer  = torch.optim.Adam(self.q_net.parameters(), lr=cfg.lr)
        self.buffer     = ReplayBuffer(cfg.buffer_capacity, (cfg.frame_stack, 84, 84), cfg.device)
        self.step_count = 0

    def epsilon(self):
        frac = min(1.0, self.step_count / self.cfg.eps_decay_steps)
        return self.cfg.eps_start + frac * (self.cfg.eps_end - self.cfg.eps_start)

    @torch.no_grad()
    def select_action(self, obs, deterministic=False):
        if not deterministic and np.random.random() < self.epsilon():
            return int(np.random.randint(self.cfg.n_actions))
        obs_t = torch.from_numpy(obs).unsqueeze(0).to(self.cfg.device)
        return int(self.q_net(obs_t).argmax(1).item())

    def learn(self):
        if len(self.buffer) < self.cfg.replay_start_size:
            return None
        batch = self.buffer.sample(self.cfg.batch_size)
        current_q = self.q_net(batch.obs).gather(1, batch.actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q = self.target_net(batch.next_obs).max(1).values
            target_q = batch.rewards + self.cfg.gamma * next_q * (1.0 - batch.dones)
        loss = F.smooth_l1_loss(current_q, target_q)
        self.optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), self.cfg.max_grad_norm)
        self.optimizer.step()
        return float(loss.item())

    def sync_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

    def save(self, path):
        torch.save({'cfg': self.cfg.__dict__, 'q_net': self.q_net.state_dict(),
                    'target_net': self.target_net.state_dict(),
                    'optimizer': self.optimizer.state_dict(),
                    'step_count': self.step_count}, path)

    def load(self, path):
        ckpt = torch.load(path, map_location=self.cfg.device, weights_only=False)
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.optimizer.load_state_dict(ckpt['optimizer'])
        self.step_count = ckpt['step_count']

print('Agent listo.')

## 6. Loop de entrenamiento

**Cadencia:**
- Cada paso del entorno: seleccionar acción → step → guardar transición.
- Cada 4 pasos: un gradiente (razón `learn_every=4`).
- Cada 1000 pasos: `target ← online`.
- Cada 5000 pasos: log.
- Cada 100 000 pasos: checkpoint a disco.

**Ojo con `terminated` vs `truncated`:** solo `terminated` colapsa el bootstrap. Un corte por tiempo NO es terminal — el estado sí tiene valor futuro definido — y debe entrar al buffer con `done=0`.

In [ ]:
from collections import deque
from pathlib import Path
import time


def train(cfg, total_steps=1_500_000, log_every=5_000, checkpoint_every=100_000,
          save_dir='saves'):
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    env = make_atari_env(cfg.env_id, seed=cfg.seed)
    agent = DQNAgent(cfg)

    episode_returns = []
    log_steps, log_running, log_eps, log_loss = [], [], [], []
    running = deque(maxlen=100)
    recent_losses = deque(maxlen=1000)

    obs, _ = env.reset(seed=cfg.seed); obs = np.asarray(obs)
    ep_return, ep_length = 0.0, 0
    t0 = time.time()

    for step in range(1, total_steps + 1):
        a = agent.select_action(obs)
        next_obs, r, term, trunc, _ = env.step(a)
        next_obs = np.asarray(next_obs)
        agent.buffer.push(obs, a, float(r), next_obs, term)  # <-- term, no term|trunc
        obs = next_obs
        ep_return += float(r); ep_length += 1
        agent.step_count = step

        if step % cfg.learn_every == 0:
            loss = agent.learn()
            if loss is not None: recent_losses.append(loss)

        if step % cfg.target_update_every == 0:
            agent.sync_target()

        if term or trunc:
            episode_returns.append(ep_return); running.append(ep_return)
            ep_return, ep_length = 0.0, 0
            obs, _ = env.reset(); obs = np.asarray(obs)

        if step % log_every == 0:
            eps = agent.epsilon()
            rmean = float(np.mean(running)) if running else float('nan')
            avg_loss = float(np.mean(recent_losses)) if recent_losses else float('nan')
            log_steps.append(step); log_running.append(rmean)
            log_eps.append(eps); log_loss.append(avg_loss)
            elapsed = time.time() - t0
            fps = step / max(elapsed, 1e-9)
            print(f'step {step:>8} | ep {len(episode_returns):>5} | '
                  f'return(mean100) {rmean:>+6.2f} | eps {eps:.3f} | '
                  f'loss {avg_loss:.4f} | buffer {len(agent.buffer):>6} | '
                  f'{fps:.0f} step/s | {elapsed/60:.1f} min', flush=True)

        if step % checkpoint_every == 0:
            agent.save(f'{save_dir}/pong_dqn_{step}.pt')
            np.savez(f'{save_dir}/history.npz',
                     episode_returns=np.array(episode_returns, dtype=np.float32),
                     log_steps=np.array(log_steps, dtype=np.int64),
                     log_running_mean=np.array(log_running, dtype=np.float32),
                     log_epsilon=np.array(log_eps, dtype=np.float32),
                     log_loss=np.array(log_loss, dtype=np.float32))

    agent.save(f'{save_dir}/pong_dqn_final.pt')
    np.savez(f'{save_dir}/history.npz',
             episode_returns=np.array(episode_returns, dtype=np.float32),
             log_steps=np.array(log_steps, dtype=np.int64),
             log_running_mean=np.array(log_running, dtype=np.float32),
             log_epsilon=np.array(log_eps, dtype=np.float32),
             log_loss=np.array(log_loss, dtype=np.float32))
    env.close()
    return agent

## 7. Entrenar 🚀

Con T4 el ritmo esperado es **~500–800 step/s**. Para 1.5 M pasos: ~35–50 min. Baja `total_steps` si necesitas terminar antes.

In [ ]:
cfg = DQNConfig()
agent = train(cfg, total_steps=1_500_000, log_every=5_000, checkpoint_every=250_000)
print('Entrenamiento completo.')

## 8. Curva de aprendizaje

In [ ]:
import matplotlib.pyplot as plt

h = np.load('saves/history.npz')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Curva de aprendizaje
returns = h['episode_returns']
ax1.plot(returns, color='steelblue', alpha=0.25, linewidth=0.6, label='Retorno por episodio')
if len(returns) >= 20:
    win = 20
    smooth = np.convolve(returns, np.ones(win)/win, mode='valid')
    ax1.plot(np.arange(win-1, len(returns)), smooth, color='firebrick', linewidth=2,
             label=f'Media móvil ({win} eps)')
ax1.axhline(0, color='gray', linestyle=':', alpha=0.6, label='Empate (0)')
ax1.axhline(-21, color='red', linestyle=':', alpha=0.4, label='Piso (-21, todos los puntos perdidos)')
ax1.axhline(21, color='green', linestyle=':', alpha=0.4, label='Techo (+21, victoria perfecta)')
ax1.set_xlabel('Episodio')
ax1.set_ylabel('Retorno (puntos anotados − puntos recibidos)')
ax1.set_title('Pong · DQN — Curva de aprendizaje')
ax1.legend(loc='lower right', framealpha=0.9, fontsize=9)
ax1.grid(True, alpha=0.3)

# Evolución de eps y loss
ax2.plot(h['log_steps'], h['log_running_mean'], color='steelblue', linewidth=2, label='Retorno medio (100 eps)')
ax2.set_xlabel('Pasos del agente')
ax2.set_ylabel('Retorno medio (100 eps)', color='steelblue')
ax2.grid(True, alpha=0.3)
ax2b = ax2.twinx()
ax2b.plot(h['log_steps'], h['log_epsilon'], color='orange', linewidth=1.5, alpha=0.8, label='ε')
ax2b.set_ylabel('ε', color='orange')
ax2.set_title('Pong · DQN — Retorno vs. ε en el tiempo')

plt.tight_layout()
plt.savefig('saves/pong_learning_curve.png', dpi=140, bbox_inches='tight')
plt.show()

print(f'Total episodios: {len(returns)}')
if len(returns) >= 100:
    print(f'Retorno medio (últimos 100 eps): {returns[-100:].mean():+.2f}')
print(f'Retorno máximo alcanzado: {returns.max():+.0f}')

## 9. Evaluación con política voraz

In [ ]:
def evaluate(agent, n_episodes=10, seed_offset=10_000):
    env = make_atari_env(seed=seed_offset)
    returns = []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed_offset + i); obs = np.asarray(obs)
        R, done = 0.0, False
        while not done:
            a = agent.select_action(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(a); obs = np.asarray(obs)
            R += float(r); done = term or trunc
        returns.append(R)
        print(f'  eval {i+1:2d}/{n_episodes}: {R:+.0f}')
    env.close()
    return np.array(returns)

eval_returns = evaluate(agent, n_episodes=10)
print(f'\n=== Evaluación (10 eps, greedy) ===')
print(f'  media: {eval_returns.mean():+.2f}')
print(f'  std:   {eval_returns.std():.2f}')
print(f'  min:   {eval_returns.min():+.0f}')
print(f'  max:   {eval_returns.max():+.0f}')
print(f'  victorias (retorno > 0): {(eval_returns > 0).sum()}/{len(eval_returns)}')

np.savez('saves/eval_results.npz', returns=eval_returns)

## 10. Grabar video del agente jugando

In [ ]:
import imageio

def record_gameplay(agent, out_path='saves/pong_gameplay.mp4', max_steps=8_000):
    env = make_atari_env(render_mode='rgb_array', seed=12_345)
    obs, _ = env.reset(seed=12_345); obs = np.asarray(obs)
    frames = []
    done = False; steps = 0
    while not done and steps < max_steps:
        frames.append(env.render())
        a = agent.select_action(obs, deterministic=True)
        obs, r, term, trunc, _ = env.step(a); obs = np.asarray(obs)
        done = term or trunc; steps += 1
    env.close()
    imageio.mimsave(out_path, frames, fps=30)
    print(f'Video guardado: {out_path} ({len(frames)} frames)')

record_gameplay(agent)

## 11. Descargar artefactos

Ejecuta la celda y baja los archivos: `pong_dqn_final.pt`, `history.npz`, `eval_results.npz`, `pong_learning_curve.png`, `pong_gameplay.mp4`.

In [ ]:
from google.colab import files
import os

for f in ['pong_dqn_final.pt', 'history.npz', 'eval_results.npz',
          'pong_learning_curve.png', 'pong_gameplay.mp4']:
    path = f'saves/{f}'
    if os.path.exists(path):
        files.download(path)
        print(f'Descargando {f}')